In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/nav_history.csv")


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/nav_history.csv'

In [2]:
import os
os.listdir("../data/raw")

['Axis_Bluechip.csv',
 'HDFC_Top100_NAV.csv',
 'ICICI_Bluechip.csv',
 'Kotak_Bluechip.csv',
 'Nippon_LargeCap.csv',
 'processed',
 'SBI_Bluechip.csv']

In [3]:
import pandas as pd
import glob

files = glob.glob("../data/raw/*.csv")

df_list = []

for file in files:
    if "processed" not in file:
        temp = pd.read_csv(file)
        temp["fund_name"] = file.split("\\")[-1].replace(".csv", "")
        df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

df.head()


,date,nav,fund_name
0,01-06-2026,6156.7532,Axis_Bluechip
1,29-05-2026,6151.1139,Axis_Bluechip
2,27-05-2026,6146.6118,Axis_Bluechip
3,26-05-2026,6144.0004,Axis_Bluechip
4,25-05-2026,6144.8478,Axis_Bluechip


In [4]:
temp["fund_name"] = file.split("\\")[-1].replace(".csv", "")

In [5]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df = df.sort_values(["fund_name", "date"])

In [6]:
df["daily_return"] = df.groupby("fund_name")["nav"].pct_change()
df.dropna(inplace=True)

In [7]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

In [8]:
df = df.sort_values(["fund_name", "date"])

In [9]:
df["daily_return"] = df.groupby("fund_name")["nav"].pct_change()

In [10]:
df = df.dropna(subset=["daily_return"])

In [11]:
import numpy as np

volatility = df.groupby("fund_name")["daily_return"].std() * np.sqrt(252)
volatility

fund_name
Axis_Bluechip      26.339781
HDFC_Top100_NAV     0.152149
ICICI_Bluechip           NaN
Kotak_Bluechip      0.154310
Nippon_LargeCap     0.167037
SBI_Bluechip        0.087177
Name: daily_return, dtype: float64

In [12]:
df[df["fund_name"] == "ICICI_Bluechip"].shape

(3305, 4)

In [13]:
df[df["fund_name"] == "ICICI_Bluechip"].head()

,date,nav,fund_name,daily_return
9960,2013-01-04,15.0702,ICICI_Bluechip,-0.000232
9959,2013-01-07,15.0913,ICICI_Bluechip,0.001400
9958,2013-01-08,15.1146,ICICI_Bluechip,0.001544
9957,2013-01-09,15.0517,ICICI_Bluechip,-0.004162
9956,2013-01-10,15.0619,ICICI_Bluechip,0.000678


In [14]:
df = df.groupby("fund_name").filter(lambda x: len(x) > 10)

In [15]:
df = df.groupby("fund_name").filter(lambda x: len(x) > 10)

volatility = df.groupby("fund_name")["daily_return"].std() * np.sqrt(252)

In [16]:
cagr = df.groupby("fund_name")["nav"].apply(
    lambda x: (x.iloc[-1] / x.iloc[0]) ** (1 / len(x)) - 1
)

In [17]:
risk_free_rate = 0.06

sharpe = (
    df.groupby("fund_name")["daily_return"].mean() * 252 - risk_free_rate
) / volatility

In [18]:
metrics = pd.DataFrame({
    "CAGR": cagr,
    "Volatility": volatility,
    "Sharpe Ratio": sharpe
})

metrics = metrics.dropna()
metrics

,CAGR,Volatility,Sharpe Ratio
fund_name,,,
Axis_Bluechip,0.001558,26.339781,0.266198
HDFC_Top100_NAV,0.000874,0.152149,1.122975
Kotak_Bluechip,0.000633,0.154310,0.722478
Nippon_LargeCap,0.000571,0.167037,0.588789
SBI_Bluechip,0.000005,0.087177,-0.628918


In [19]:
metrics.to_csv("../data/processed/performance_metrics.csv")
print("Saved successfully")

Saved successfully


In [20]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../data/db/bluestock_mf.db")

In [21]:
df.to_sql("fact_nav", engine, if_exists="replace", index=False)
metrics.to_sql("fact_performance", engine, if_exists="replace", index=False)

5

In [22]:
pd.read_sql("SELECT COUNT(*) FROM fact_nav", engine)

,COUNT(*)
0,19786
